# Clonamos el repositorio con los modelos y herramientas

In [1]:
!git clone https://github.com/dannasalazar11/Msc_thesis.git

Cloning into 'Msc_thesis'...
remote: Enumerating objects: 554, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 554 (delta 62), reused 0 (delta 0), pack-reused 427 (from 1)
Receiving objects: 100% (554/554), 52.73 MiB | 43.44 MiB/s, done.
Resolving deltas: 100% (339/339), done.


In [2]:
import sys
sys.path.append('/kaggle/working/Msc_thesis')

from gmrrnet_adhd.utils import get_segmented_data
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import random
import os

# Establecer semilla
seed = 42

# Semillas para módulos principales
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

2025-11-26 19:13:10.296619: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764184390.662400      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764184390.780719      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import numpy as np
import random
from collections import defaultdict
from copy import deepcopy

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    cohen_kappa_score,
    roc_auc_score
)


def train_L24O_cv(model_builder, X, y, sbjs, model_args, compile_args, folds, model_name=''):
    all_fold_metrics = []
    models = {}

    for fold, (train_subjects, test_subjects) in enumerate(folds):
        print("-" * 50)
        print(f"Fold {fold+1}/{len(folds)}. Test subjects: {test_subjects}")
        print("-" * 50)

        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        test_idx = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        sbjs_test = [sbjs[i] for i in test_idx]

        # --- Build and Compile Model for each fold ---
        tf.keras.backend.clear_session() #<-- Clear session to prevent any state leakage
        
        # Re-set seeds for each fold for perfect reproducibility of weight initialization
        np.random.seed(seed + fold)
        random.seed(seed + fold)
        tf.random.set_seed(seed + fold)

        model = model_builder(**model_args)
        # Use a deepcopy to prevent the optimizer state from carrying over
        compile_args_local = deepcopy(compile_args)
        if callable(compile_args_local["optimizer"]):
            compile_args_local["optimizer"] = compile_args_local["optimizer"]()  # <-- aquí se reinicia
        model.compile(**compile_args_local)

    

        # --- Train the Model ---
        model.fit(
            X_train, y_train,
            epochs=100,  #<-- Increased epochs to give LR scheduler more time to work
            validation_data=(X_test, y_test),
            verbose=0, #<-- Verbose=2 gives one line per epoch, cleaner log
            batch_size=16
        )

        # --- Predictions and Evaluation ---
        y_pred_probs = model.predict(X_test)
        print(y_pred_probs.shape)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.argmax(y_test, axis=1)

        # Overall fold metrics
        fold_metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'kappa': cohen_kappa_score(y_true, y_pred),
            'auc': roc_auc_score(y_true, y_pred_probs[:, 1]) # Use probabilities for AUC
        }
        print(f"\nFold {fold+1} Metrics: {fold_metrics}")
        all_fold_metrics.append(fold_metrics)
        models[fold] = model

        # Accuracy por sujeto de test
        subject_correct = defaultdict(list)
        for yt, yp, sbj in zip(y_true, y_pred, sbjs_test):
            subject_correct[sbj].append(int(yt == yp))

        subject_accuracies = {
            sbj: np.mean(subject_correct[sbj]) for sbj in subject_correct
        }

        print("Average accuracy per test subject:")
        for sbj in test_subjects:
            acc_sbj = subject_accuracies.get(sbj, None)
            if acc_sbj is not None:
                print(f"  {sbj}: {acc_sbj:.4f}")
                
        
    # --- Final Comprehensive Report ---
    print("\n" + "="*50)
    print("Cross-Validation Final Results")
    print("="*50)
    
    # Calculate mean and std dev for each metric
    mean_metrics = {}
    for key in all_fold_metrics[0].keys():
        values = [f[key] for f in all_fold_metrics]
        mean_metrics[f'mean_{key}'] = np.mean(values)
        mean_metrics[f'std_{key}'] = np.std(values)

    print("Individual Fold Accuracies:")
    for i, f in enumerate(all_fold_metrics):
        print(f"  Fold {i+1}: {f['accuracy']:.4f}")
        
    print("\nAverage Performance across all folds:")
    for key, value in mean_metrics.items():
        print(f"  {key}: {value:.4f}")
        
    return all_fold_metrics

# Importar base de datos segmentada (Segmentos de 4 seg con translape del 50%, es decir, de 2 seg)

In [4]:
X, y, sbjs = get_segmented_data()
X.shape, y.shape, len(sbjs)

((8213, 19, 512), (8213, 2), 8213)

# Importamos el modelo y definimos los hiperparámetros

In [5]:
from tensorflow.keras.losses import CategoricalCrossentropy, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from gmrrnet_adhd.models.cnn_lstm_eegnet import cnn_lstm_eegnet

model_name = 'CNN_LSTM_EEGNet'
model_args = {
    "n_channels": 19,      # EEG electrodes
    "n_times": 512,        # time points per trial
    "n_classes": 2,        # binary task
    "temporal_kernel": 25, # ≈200 ms at 128 Hz
    "pool_size": 20,
    "pool_stride": 10
}

compile_args = {
    'loss': CategoricalCrossentropy(),
    'optimizer': lambda: Adam(1e-2),  # función que retorna un nuevo optimizador
    'metrics': ['categorical_accuracy']
}


model = cnn_lstm_eegnet(**model_args)

model.summary()

I0000 00:00:1764184412.061380      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764184412.061981      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "cnn_lstm_eegnet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 19, 512, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast (Cast)               │ (None, 19, 512, 1)     │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 19, 512, 50)    │          1,250 │ cast[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 19, 512, 50)    │            200 │ conv2d[0][0]           │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 1, 512, 100)    │              0 │ batch_normalization[0… │
│                           │                        │                │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ depthwise_conv2d          │ (None, 1, 512, 100)    │          1,900 │ activation[0][0]       │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 1, 512, 100)    │            400 │ depthwise_conv2d[0][0] │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling2d         │ (None, 1, 50, 100)     │              0 │ activation[1][0]       │
│ (AveragePooling2D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 1, 50, 100)     │              0 │ average_pooling2d[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ permute (Permute)         │ (None, 50, 1, 100)     │              0 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed          │ (None, 50, 100)        │              0 │ permute[0][0]          │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ (None, 50, 10)         │          4,440 │ time_distributed[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ (None, 10)             │            840 │ lstm[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 2)              │             22 │ lstm_1[0][0]           │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 9,052 (35.36 KB)

 Trainable params: 8,752 (34.19 KB)

 Non-trainable params: 300 (1.17 KB)

# Resultados - Leave 24 Subjects Out

In [6]:
import os

import pickle

with open("/kaggle/input/ieee-tdah-control-database/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [7]:
import numpy as np

results = {}

for i in range(10):
    result = train_L24O_cv(cnn_lstm_eegnet, X, y, sbjs, model_args, compile_args, folds)
    results[i] = result

--------------------------------------------------
Fold 1/5. Test subjects: ['v28p', 'v274', 'v1p', 'v231', 'v22p', 'v29p', 'v206', 'v238', 'v31p', 'v35p', 'v177', 'v200', 'v112', 'v113', 'v48p', 'v140', 'v131', 'v125', 'v55p', 'v143', 'v43p', 'v305', 'v134', 'v114']
--------------------------------------------------


E0000 00:00:1764184420.657009      20 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1764184421.095539      74 cuda_dnn.cc:529] Loaded cuDNN version 90300


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
(1457, 2)

Fold 1 Metrics: {'accuracy': 0.7913520933424846, 'recall': 0.7961516531484587, 'precision': 0.8064803851459342, 'kappa': 0.5861885978762325, 'auc': 0.8470573353270803}
Average accuracy per test subject:
  v28p: 0.0000
  v274: 1.0000
  v1p: 1.0000
  v231: 1.0000
  v22p: 1.0000
  v29p: 0.9570
  v206: 0.0000
  v238: 1.0000
  v31p: 1.0000
  v35p: 1.0000
  v177: 0.2188
  v200: 1.0000
  v112: 0.9836
  v113: 1.0000
  v48p: 1.0000
  v140: 1.0000
  v131: 1.0000
  v125: 1.0000
  v55p: 1.0000
  v143: 0.6949
  v43p: 0.0000
  v305: 1.0000
  v134: 1.0000
  v114: 1.0000
--------------------------------------------------
Fold 2/5. Test subjects: ['v18p', 'v39p', 'v234', 'v32p', 'v190', 'v6p', 'v254', 'v204', 'v24p', 'v183', 'v246', 'v219', 'v298', 'v41p', 'v47p', 'v308', 'v52p', 'v300', 'v59p', 'v299', 'v302', 'v51p', 'v109', 'v127']
--------------------------------------------------


E0000 00:00:1764185088.466525      20 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step
(1668, 2)

Fold 2 Metrics: {'accuracy': 0.776378896882494, 'recall': 0.7654664379498854, 'precision': 0.7792445714495961, 'kappa': 0.5393970801622789, 'auc': 0.7851721278384953}
Average accuracy per test subject:
  v18p: 1.0000
  v39p: 1.0000
  v234: 1.0000
  v32p: 1.0000
  v190: 1.0000
  v6p: 0.2388
  v254: 1.0000
  v204: 0.0000
  v24p: 1.0000
  v183: 1.0000
  v246: 0.9756
  v219: 1.0000
  v298: 0.0000
  v41p: 1.0000
  v47p: 1.0000
  v308: 1.0000
  v52p: 1.0000
  v300: 1.0000
  v59p: 0.0952
  v299: 1.0000
  v302: 0.9118
  v51p: 0.3667
  v109: 0.4098
  v127: 0.0000
--------------------------------------------------
Fold 3/5. Test subjects: ['v215', 'v3p', 'v209', 'v37p', 'v213', 'v15p', 'v284', 'v181', 'v19p', 'v34p', 'v263', 'v244', 'v138', 'v121', 'v46p', 'v54p', 'v120', 'v310', 'v147', 'v50p', 'v56p', 'v107', 'v297', 'v108']
--------------------------------------------------


...

56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
(1768, 2)

Fold 3 Metrics: {'accuracy': 0.7839366515837104, 'recall': 0.769205313477846, 'precision': 0.776050824335585, 'kappa': 0.5443919906203747, 'auc': 0.8231700726566722}
Average accuracy per test subject:
  v215: 1.0000
  v3p: 1.0000
  v209: 1.0000
  v37p: 1.0000
  v213: 1.0000
  v15p: 0.0000
  v284: 1.0000
  v181: 1.0000
  v19p: 0.9888
  v34p: 1.0000
  v263: 1.0000
  v244: 1.0000
  v138: 0.0000
  v121: 1.0000
  v46p: 1.0000
  v54p: 1.0000
  v120: 1.0000
  v310: 0.0000
  v147: 0.1296
  v50p: 1.0000
  v56p: 1.0000
  v107: 1.0000
  v297: 0.0000
  v108: 1.0000
--------------------------------------------------
Fold 4/5. Test subjects: ['v227', 'v8p', 'v236', 'v14p', 'v196', 'v27p', 'v33p', 'v179', 'v173', 'v10p', 'v265', 'v20p', 'v57p', 'v45p', 'v111', 'v115', 'v53p', 'v118', 'v123', 'v44p', 'v149', 'v303', 'v116', 'v151']
--------------------------------------------------


E0000 00:00:1764215867.389455      20 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
(1687, 2)

Fold 4 Metrics: {'accuracy': 0.8233550681683461, 'recall': 0.823679167885808, 'precision': 0.8196093982353525, 'kappa': 0.6422943209778158, 'auc': 0.8923254332971894}
Average accuracy per test subject:
  v227: 1.0000
  v8p: 1.0000
  v236: 1.0000
  v14p: 0.5672
  v196: 0.9608
  v27p: 0.5856
  v33p: 1.0000
  v179: 1.0000
  v173: 1.0000
  v10p: 1.0000
  v265: 1.0000
  v20p: 0.3066
  v57p: 1.0000
  v45p: 0.8780
  v111: 1.0000
  v115: 1.0000
  v53p: 0.7945
  v118: 1.0000
  v123: 1.0000
  v44p: 0.0000
  v149: 0.5077
  v303: 1.0000
  v116: 1.0000
  v151: 0.6125
--------------------------------------------------
Fold 5/5. Test subjects: ['v279', 'v30p', 'v288', 'v286', 'v250', 'v12p', 'v38p', 'v25p', 'v21p', 'v40p', 'v198', 'v270', 'v117', 'v306', 'v309', 'v110', 'v42p', 'v58p', 'v307', 'v133', 'v304', 'v129', 'v49p', 'v60p']
--------------------------------------------------


E0000 00:00:1764216517.502633      20 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
(1633, 2)

Fold 5 Metrics: {'accuracy': 0.863441518677281, 'recall': 0.8604584140135993, 'precision': 0.8833227981458069, 'kappa': 0.7251055503678883, 'auc': 0.9390297353687276}
Average accuracy per test subject:
  v279: 0.9740
  v30p: 1.0000
  v288: 1.0000
  v286: 1.0000
  v250: 1.0000
  v12p: 0.7313
  v38p: 1.0000
  v25p: 1.0000
  v21p: 1.0000
  v40p: 0.9740
  v198: 1.0000
  v270: 1.0000
  v117: 0.1010
  v306: 1.0000
  v309: 1.0000
  v110: 0.0000
  v42p: 1.0000
  v58p: 1.0000
  v307: 1.0000
  v133: 1.0000
  v304: 1.0000
  v129: 1.0000
  v49p: 1.0000
  v60p: 0.0000

Cross-Validation Final Results
Individual Fold Accuracies:
  Fold 1: 0.6836
  Fold 2: 0.4167
  Fold 3: 0.7839
  Fold 4: 0.8234
  Fold 5: 0.8634

Average Performance across all folds:
  mean_accuracy: 0.7142
  std_accuracy: 0.1603
  mean_recall: 0.7234
  std_recall: 0.1383
  mean_precision: 0.6895
  std_precision: 0.2284
  mean_kappa: 0.4476
  std_kappa: 0.2750
  mean_auc: 0.7870
  st

In [8]:
for i in range(10):
    result = results[i]
    accs = []
    for r in result:
        accs.append(r['accuracy'])
    
    print(i, '->', np.mean(accs))

0 -> 0.8094693700214266
1 -> 0.8342416671413233
2 -> 0.8432442904092179
3 -> 0.8130521562256361
4 -> 0.7726257604944395
5 -> 0.8191600496728823
6 -> 0.8287924597618611
7 -> 0.7658664494966094
8 -> 0.820052726535954
9 -> 0.7141992672237307
